# 05 · Agents intro — a hello agent in Agno + Gemini

**Workshop:** AI for Actuaries — From Foundations to AI Agents  
**Session / Part:** S2.P3  
**Slides covered:** S2.P3.5 – S2.P3.11  
**Author:** Dr Rohan Yashraj Gupta (FIA, FIAI), with Satya Sai Mudigonda and Sai Krishna Vadali  
**Workshop date:** 15 May 2026 · Hilton near Airport, Mumbai  

## What this notebook does
Builds the simplest possible Agno agent — one Gemini model, one Python tool, one user query — and prints the full ReAct trace so you can see exactly what the agent reasoned, which tool it called, what came back, and how it answered. The tool is `lookup_idv()`, a deliberately tiny in-notebook lookup table for ABC General's motor IDV. Same shape as the rating-logic-interpreter agent we'll build properly in Part 4.

## Prerequisites
- Google account (for Colab).
- A Gemini API key — free tier is fine. Get one at [Google AI Studio](https://aistudio.google.com/apikey) and store it in Colab Secrets as `GOOGLE_API_KEY` (left sidebar key icon → New secret → name `GOOGLE_API_KEY`, value your key, toggle 'Notebook access' on).
- No local install required.

## How to run
Top menu → Runtime → Run all. The first cell installs dependencies; subsequent cells run unattended once the API key is set.

---

*Illustrative example. ABC Insurer is a hypothetical entity. Figures are for teaching only.*

## 1. Install dependencies

Two packages: **`agno`** (the agent framework) and **`google-genai`** (Google's Gemini SDK that Agno calls under the hood).

> **Version pin policy:** the pins below are placeholders. Rohan to re-confirm exact versions on a fresh Colab CPU runtime at notebook-finalisation time (per `00_citations.md` items D1 and D2). If you're running this notebook standalone before then, the unpinned `pip install` should still work — Agno is on rapid release cadence.

In [1]:
# Installing all the required packages
!pip install -q agno google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 45.8 MB/s eta 0:00:00


## 2. Standard imports and reproducibility

In [2]:
# === Standard imports ===
import os
import numpy as np
import pandas as pd

# Reproducibility (no randomness in this notebook, but kept for consistency)
SEED = 42
np.random.seed(SEED)

# Display
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Notebook-specific imports
# -------------------------------------------------------------
from agno.agent import Agent
from agno.models.google import Gemini

print("Imports OK.")

Imports OK.


## 3. API key setup

Reads `GEMINI_API_KEY` from Colab Secrets. Falls back to an environment variable if you're running this outside Colab.

In [3]:
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")
except (ImportError, Exception):
    if "GOOGLE_API_KEY" not in os.environ:
        raise RuntimeError(
            "GOOGLE_API_KEY is not set. "
            "Add it via Colab Secrets (left sidebar key icon, name: GOOGLE_API_KEY) "
            "or export it as an environment variable."
        )

print("API key loaded.")

API key loaded.


## 4. The lookup table — ABC General motor IDV (hypothetical)

A deliberately tiny in-notebook table. In production this would be a CSV, a database, or a rating engine API. For today, eight rows is all we need to make the agent reason.

In [4]:
# Hypothetical IDV reference table for ABC General — illustrative only.
# Schema matches abc_motor_2024.csv (see 00_personas_and_datasets.md).
IDV_TABLE = pd.DataFrame(
    [
        # make,      segment,     0-yr IDV (INR), depreciation_per_year
        ("Maruti",   "Hatchback",     650_000, 0.10),
        ("Maruti",   "Sedan",         900_000, 0.10),
        ("Maruti",   "SUV",         1_100_000, 0.10),
        ("Hyundai",  "Hatchback",     720_000, 0.09),
        ("Hyundai",  "Sedan",       1_050_000, 0.09),
        ("Tata",     "SUV",         1_400_000, 0.11),
        ("Mahindra", "SUV",         1_650_000, 0.11),
        ("Honda",    "Sedan",       1_250_000, 0.08),
    ],
    columns=["vehicle_make", "vehicle_segment", "base_idv_inr", "annual_depreciation"],
)
IDV_TABLE

,vehicle_make,vehicle_segment,base_idv_inr,annual_depreciation
0,Maruti,Hatchback,650000,0.10
1,Maruti,Sedan,900000,0.10
2,Maruti,SUV,1100000,0.10
3,Hyundai,Hatchback,720000,0.09
4,Hyundai,Sedan,1050000,0.09
5,Tata,SUV,1400000,0.11
6,Mahindra,SUV,1650000,0.11
7,Honda,Sedan,1250000,0.08


## 5. The tool — a plain Python function

Three things to notice:

1. **It's just a Python function** — no decorators, no special framework wrapping. Agno reads the type hints and the docstring to build a function-calling spec for Gemini.
2. **The docstring is the agent's user manual.** Write it for the model, not for a human reader. Be specific about what arguments mean and what the function returns.
3. **It returns a structured string.** Tool outputs go back to the model as text — return something the model can quote in its answer.

We also handle the unhappy path: unknown make/segment combos return a clear error message rather than raising. The agent will see the error and reason about it.

In [5]:
def lookup_idv(vehicle_make: str, vehicle_segment: str, vehicle_age_years: int) -> str:
    """Look up the rough Insured Declared Value (IDV) in INR for an ABC General motor policy.

    Use this when the user asks about IDV, replacement value, or sum insured for a private car.
    Applies straight-line depreciation at the manufacturer's published annual rate. Figures are
    illustrative and based on ABC General's reference table — not market quotes.

    Args:
        vehicle_make: Manufacturer, e.g. 'Maruti', 'Hyundai', 'Tata', 'Mahindra', 'Honda'.
                      Title-case match is enforced.
        vehicle_segment: One of 'Hatchback', 'Sedan', 'SUV'.
        vehicle_age_years: Integer age of the vehicle in years (0 = brand new). 0 to 15 supported.

    Returns:
        A string describing the looked-up IDV in INR, including make, segment, age, and the
        depreciation applied. If the make/segment is not in the reference table, returns an
        explanatory error string starting with 'NO_MATCH:'.
    """
    # Normalise inputs — actuaries are forgiving, agents shouldn't be
    make = vehicle_make.strip().title()
    segment = vehicle_segment.strip().title()

    if not (0 <= vehicle_age_years <= 15):
        return (
            f"NO_MATCH: vehicle_age_years={vehicle_age_years} is outside the supported range "
            f"(0-15). Ask the user to confirm."
        )

    row = IDV_TABLE[
        (IDV_TABLE["vehicle_make"] == make) & (IDV_TABLE["vehicle_segment"] == segment)
    ]
    if row.empty:
        return (
            f"NO_MATCH: no entry for make='{make}', segment='{segment}'. "
            f"Reference table covers: "
            f"{list(IDV_TABLE[['vehicle_make','vehicle_segment']].itertuples(index=False, name=None))}."
        )

    base = float(row["base_idv_inr"].iloc[0])
    depr = float(row["annual_depreciation"].iloc[0])
    factor = max(1 - depr * vehicle_age_years, 0.20)  # floor at 20% of base, IRDAI-style
    idv = round(base * factor, -3)  # round to nearest thousand

    return (
        f"IDV for a {vehicle_age_years}-year-old {make} {segment} (ABC General reference table): "
        f"INR {idv:,.0f}. Base INR {base:,.0f}, annual depreciation {depr:.0%}, "
        f"applied factor {factor:.2f}."
    )


# Smoke test — call the tool directly so we know it works before the agent does
print(lookup_idv("Maruti", "Hatchback", 5))
print(lookup_idv("Ferrari", "Coupe", 2))  # should return NO_MATCH

IDV for a 5-year-old Maruti Hatchback (ABC General reference table): INR 325,000. Base INR 650,000, annual depreciation 10%, applied factor 0.50.
NO_MATCH: no entry for make='Ferrari', segment='Coupe'. Reference table covers: [('Maruti', 'Hatchback'), ('Maruti', 'Sedan'), ('Maruti', 'SUV'), ('Hyundai', 'Hatchback'), ('Hyundai', 'Sedan'), ('Tata', 'SUV'), ('Mahindra', 'SUV'), ('Honda', 'Sedan')].


## 6. The agent — Gemini brain, one tool, ReAct loop

Four lines of configuration is all it takes:

- `model=Gemini(...)` — the LLM that reasons. Pinned to `gemini-2.5-flash` (free-tier-friendly as of workshop date).
- `tools=[lookup_idv]` — the Python function from §5. Agno introspects it.
- `instructions=...` — the system prompt. Tell the agent what role it plays and what it must NOT do.
- `debug_mode=True`, `debug_level=2` — print the full ReAct trace so we can watch reasoning, tool call, and observation stream past.

In [6]:
# Pin the model identifier explicitly — never let the SDK pick.
# 'gemini-2.5-flash' = current free-tier-eligible flash model as of May 2026.
# GEMINI_MODEL_ID = "gemini-2.5-flash"
MODEL_ID = "gemini-3.1-flash-lite"

agent = Agent(
    model=Gemini(id=MODEL_ID),
    tools=[lookup_idv],
    instructions=[
        "You are a junior pricing assistant at ABC General Insurance.",
        "When the user asks about IDV, replacement value, or sum insured for a motor policy, "
        "call the lookup_idv tool — do NOT guess from memory.",
        "If the tool returns a string starting with 'NO_MATCH:', explain to the user that the "
        "vehicle is outside ABC General's reference table and ask for clarification.",
        "Always show the user the figures the tool returned and label them as illustrative.",
    ],
    # debug_mode=True,   # print the full ReAct trace (reasoning + tool calls + observations)
    # debug_level=2,     # 2 = verbose: include the model's intermediate reasoning steps
    markdown=True,
)

print("Agent ready. Model:", MODEL_ID)

Agent ready. Model: gemini-3.1-flash-lite


## 7. Run the agent — one query, one trace

The query: **"What's the rough IDV for a 5-year-old Maruti hatchback?"**

Watch the trace:

1. **Reason** — the model decides it needs to call `lookup_idv`.
2. **Act** — it emits a structured tool call with `vehicle_make='Maruti'`, `vehicle_segment='Hatchback'`, `vehicle_age_years=5`.
3. **Observe** — the runtime executes the function and feeds the return string back.
4. **Answer** — the model composes a human-readable response using the figure it just looked up.

That four-step loop is the entire ReAct pattern from slide S2.P3.4.

In [7]:
user_query = "What's the rough IDV for a 5-year-old Maruti hatchback?"
agent.print_response(user_query, stream=True, show_full_reasoning=False)

Output()

## 8. A second query — the unhappy path

Now ask about a vehicle that's not in the reference table. The tool will return a `NO_MATCH:` string, and the agent should reason its way into asking for clarification rather than fabricating a number.

**This is the moment that distinguishes an agent from a chatbot:** the agent knows it failed, and it knows what to do about it.

In [ ]:
agent.print_response(
    "What's the IDV for a 3-year-old Ferrari Roma?",
    stream=True,
    show_full_reasoning=True,
)

Output()

## Wrap-up

You should now be able to:
- Wire an Agno agent to Gemini in four lines of configuration.
- Expose a Python function as a tool — including a useful docstring and a sensible unhappy path.
- Read a ReAct trace and identify Reason / Act / Observe / Answer in it.

**Where to next:** open `06_vibe_agent.ipynb` for the full Pricing Logic Explainer Agent — three tools, three LOB extensions (Health and Life), and the guardrails added on top.

**Companion slides:** S2.P3.5 – S2.P3.11 of the deck.